# Module 2 -- Step 2: Explore the LLM Gateway

The LLM Gateway stack (`workshop-llm-gateway-stack`) was automatically deployed
when your workshop environment was provisioned. This notebook verifies the stack,
captures outputs, and confirms the gateway is healthy.

## 2.1 Install Dependencies

## 2.2 Configuration

In [ ]:
import boto3
import json
import time

AWS_REGION = boto3.session.Session().region_name or "us-west-2"
STACK_NAME = "workshop-llm-gateway-stack"
print(f"Region:     {AWS_REGION}")
print(f"Stack name: {STACK_NAME}")

## 2.3 Verify the Pre-Deployed Stack

The `workshop-llm-gateway-stack` was deployed automatically by Workshop Studio.
Confirm it exists and is in `CREATE_COMPLETE` state.

In [ ]:
cfn = boto3.client("cloudformation", region_name=AWS_REGION)
try:
    resp = cfn.describe_stacks(StackName=STACK_NAME)
except cfn.exceptions.ClientError as e:
    raise RuntimeError(
        f"Stack {STACK_NAME} not found — Workshop Studio may not have finished "
        f"provisioning. Retry in 2 minutes. Underlying error: "
        f"{e.response['Error']['Message']}"
    ) from e
stack = resp["Stacks"][0]
status = stack["StackStatus"]
print(f"Stack:  {STACK_NAME}")
print(f"Status: {status}")
assert status in ("CREATE_COMPLETE", "UPDATE_COMPLETE"), (
    f"Stack {STACK_NAME} is in status {status}. Expected CREATE_COMPLETE or UPDATE_COMPLETE. "
    f"Check the CloudFormation console before continuing."
)

## 2.4 Capture Stack Outputs

Read the key outputs from the pre-deployed stack: the gateway URL and admin API key.

In [ ]:
outputs = {o["OutputKey"]: o["OutputValue"] for o in stack["Outputs"]}

LLM_GATEWAY_URL = outputs["ProxyUrl"].rstrip("/")
# The stack publishes the admin key secret ARN as `AdminKeySecretArn`.
ADMIN_KEY_ARN = outputs["AdminKeySecretArn"]

print(f"LLM Gateway URL:    {LLM_GATEWAY_URL}")
print(f"Admin Key ARN:      {ADMIN_KEY_ARN}")

# Retrieve the admin key from Secrets Manager
sm = boto3.client("secretsmanager", region_name=AWS_REGION)
raw_secret = sm.get_secret_value(SecretId=ADMIN_KEY_ARN)["SecretString"]
# Secret may be a plain string or a JSON blob ({"master_key": "..."})
# ("master_key" is the upstream LiteLLM field name — not renamable)
try:
    parsed = json.loads(raw_secret)
    key_val = parsed.get("master_key") if isinstance(parsed, dict) else raw_secret
except (json.JSONDecodeError, TypeError):
    key_val = raw_secret
key_val = key_val or raw_secret
ADMIN_KEY = key_val if key_val.startswith("sk-") else f"sk-{key_val}"
print(f"Admin Key:          (retrieved from Secrets Manager)")

## 2.5 Wait for Gateway Health Check

The ECS service may take a minute or two after initial provisioning. This cell
polls the health endpoint until the gateway responds.

In [ ]:
import requests

health_url = f"{LLM_GATEWAY_URL}/health/liveliness"
print("Waiting for LLM Gateway to be healthy...")
for attempt in range(30):
    try:
        resp = requests.get(health_url, timeout=5)
        if resp.ok:
            print(f"\nGateway is healthy! (attempt {attempt + 1})")
            print(json.dumps(resp.json(), indent=2))
            break
    except requests.ConnectionError:
        pass
    print(f"  Attempt {attempt + 1}/30 - waiting 10s...")
    time.sleep(10)
else:
    raise RuntimeError(
        f"LLM Gateway {health_url} did not become healthy in 5 minutes. "
        f"Check the ECS service state in the CloudFormation console."
    )

## 2.6 Verify ECS Service

Confirm that the ECS service backing the LLM Gateway is running
with the expected number of tasks.

In [ ]:
ecs = boto3.client("ecs", region_name=AWS_REGION)
clusters = ecs.list_clusters()["clusterArns"]
print(f"ECS Clusters: {len(clusters)}")
found_service = False
for cluster in clusters:
    services = ecs.list_services(cluster=cluster)["serviceArns"]
    for svc in services:
        if "llm-gateway" in svc.lower() or "litellm" in svc.lower():
            found_service = True
            detail = ecs.describe_services(cluster=cluster, services=[svc])["services"][0]
            print(f"\nService: {detail['serviceName']}")
            print(f"  Status:       {detail['status']}")
            print(f"  Running:      {detail['runningCount']}")
            print(f"  Desired:      {detail['desiredCount']}")
assert found_service, "Did not find an ECS service matching 'llm-gateway' or 'litellm'"

## 2.7 Save State for Subsequent Notebooks

Persist the gateway URL, API key, and other variables so that later
notebooks (Step 3, etc.) can load them without re-querying CloudFormation.

In [ ]:
state = {
    "LLM_GATEWAY_URL": LLM_GATEWAY_URL,
    "ADMIN_KEY": ADMIN_KEY,
    "ADMIN_KEY_ARN": ADMIN_KEY_ARN,
    "AWS_REGION": AWS_REGION,
    "STACK_NAME": STACK_NAME,
}
import os
state_path = os.path.abspath(".state.json")
with open(state_path, "w") as f:
    json.dump(state, f, indent=2)
print(f"State saved to {state_path}")

## 2.8 Log in to the LiteLLM Admin UI (optional)

The LiteLLM proxy ships with a built-in admin UI for browsing keys, budgets, and spend. Open it in your browser:

- **URL:** `{LLM_GATEWAY_URL}/ui` — the cell below prints the exact link.
- **Username:** `admin`
- **Password:** the **admin key** retrieved above (the same `ADMIN_KEY` value stored in `.state.json`). Treat it like a shared admin password — do not commit it.

![LiteLLM Admin UI login](/static/img/module-2/litellm-admin-ui.png)

After logging in you will land on the Keys tab. The team, virtual keys, budgets, and guardrails you create in later steps show up here in real time, which is useful when debugging.


In [ ]:
print(f"Admin UI: {LLM_GATEWAY_URL}/ui")
print(f"Username: admin")
print(f"Password: (admin key — see the ADMIN_KEY variable above; not reprinted here)")

---

## Next Step

Proceed to **Step 3** to register Bedrock models and create virtual keys
with team-based budgets for cost governance.

Open **`step-3-virtual-keys.ipynb`** to continue.